# Colab runner — Semarang Batik (Kelompok 5, Track A)

Runs the frozen experiment matrix on a GPU. The split manifest committed in `data/manifests/split.csv` is **reused, never regenerated**, so results are comparable everywhere.

Runtime → Change runtime type → GPU. Runs (`artifacts/runs`) and the resized image cache live on Google Drive, so after a disconnect re-run all cells: finished runs are skipped, an interrupted run resumes from `last.pt`, and the 13 GB download is not repeated.

In [14]:
from google.colab import drive
drive.mount('/content/drive')
REPO_URL = 'https://github.com/Universitas-Cakrawala/semarang-batik-fine-grained-transfer-learning.git'
BRANCH = 'titan/model-training'  # branch that holds this code; switch to main after merging
# Reused runtime: pull, otherwise Colab keeps running stale code
!if [ -d /content/repo ]; then git -C /content/repo pull --ff-only; else git clone -b $BRANCH $REPO_URL /content/repo; fi
%cd /content/repo
!pip -q install -e .
!nvidia-smi --query-gpu=name,memory.total --format=csv

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Already up to date.
/content/repo
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for batik (pyproject.toml) ... done
name, memory.total [MiB]
Tesla T4, 15360 MiB


In [15]:
# Persist runs and the image cache on Drive (symlinks) so a disconnect never loses work
DRIVE = '/content/drive/MyDrive/batik'
!mkdir -p $DRIVE/runs $DRIVE/processed data
!rm -rf artifacts/runs data/processed && ln -s $DRIVE/runs artifacts/runs && ln -s $DRIVE/processed data/processed

In [16]:
# One-off: ~13 GB download + cache build (EXIF-transposed, short side 320 px). Skipped when the cache exists.
import pathlib
if not any(pathlib.Path('data/processed/short320').glob('*/*.jpg')):
    !python -m batik download --delete-zip
    !python -m batik audit --workers 2
    !git diff --stat data/manifests   # expected: no changes (audit is deterministic)
!python -m batik verify-split   # must print OK: the committed split is used as-is

[split] manifest checksum OK


In [17]:
!python -m batik sanity

[split] manifest checksum OK
[sanity] overfit_custom_cnn: {'model': 'custom_cnn', 'pretrained': False, 'n_images': 32, 'epochs_used': 19, 'final_train_acc': 1.0, 'first_loss': 1.3983056545257568, 'last_loss': 0.050188157707452774, 'passed': True}
[sanity] overfit_efficientnet_b0: {'model': 'efficientnet_b0', 'pretrained': False, 'n_images': 32, 'epochs_used': 40, 'final_train_acc': 0.875, 'first_loss': 2.7734222412109375, 'last_loss': 0.08105818182229996, 'passed': False}
[sanity] overfit_efficientnet_b0_pretrained: {'model': 'efficientnet_b0', 'pretrained': True, 'n_images': 32, 'epochs_used': 3, 'final_train_acc': 1.0, 'first_loss': 1.9114856719970703, 'last_loss': 0.24353019893169403, 'passed': True}
{
  "labels": true,
  "loader": true,
  "split_reproducible": false,
  "transforms": true,
  "checkpoint_roundtrip": true,
  "overfit_custom_cnn": true,
  "overfit_efficientnet_b0": false,
  "overfit_efficientnet_b0_pretrained": true,
  "all_passed": false
}


In [18]:
# G1-G3 x 3 seeds, G4, G1g-G3g, then G5. Re-run this cell after a disconnect to resume.
!python -m batik matrix --num-workers 2

[matrix] order: G1/s42, G2/s42, G3/s42, G1/s43, G2/s43, G3/s43, G1/s44, G2/s44, G3/s44, G4/s42, G4/s43, G4/s44, G1g/s42, G2g/s42, G3g/s42, G1g/s43, G2g/s43, G3g/s43, G1g/s44, G2g/s44, G3g/s44, G5/s42, G5/s43, G5/s44
[train] G1_customcnn-scratch_augon_s42 already complete -> skipping
[train] G2_effb0-scratch_augon_s42 already complete -> skipping
[train] G3_effb0-pt_augon_s42 already complete -> skipping
[train] G1_customcnn-scratch_augon_s43 already complete -> skipping
[train] G2_effb0-scratch_augon_s43 already complete -> skipping
[train] G3_effb0-pt_augon_s43 already complete -> skipping
[train] G1_customcnn-scratch_augon_s44 already complete -> skipping
[train] G2_effb0-scratch_augon_s44 already complete -> skipping
[train] G3_effb0-pt_augon_s44 already complete -> skipping
[train] G4_effb0-pt_augoff_s42 already complete -> skipping
[train] G4_effb0-pt_augoff_s43 already complete -> skipping
[train] G4_effb0-pt_augoff_s44 already complete -> skipping
[train] G1g_customcnn-scratch_a

In [19]:
!python -m batik aggregate
!python -m batik robustness
!python -m batik errors
!python -m batik release
!cat artifacts/metrics/tables.md
# Keep the small, committable outputs on Drive as well
!mkdir -p $DRIVE/outputs && cp -r artifacts/metrics artifacts/error_analysis artifacts/release figures $DRIVE/outputs/

{
  "description": "G3 EfficientNet-B0 ImageNet + aug ON, test Macro F1 (mean \u00b1 std over seeds)",
  "test_macro_f1_mean": 1.0,
  "test_macro_f1_std": 0.0,
  "seeds": [
    42,
    43,
    44
  ],
  "per_seed": {
    "42": 1.0,
    "43": 1.0,
    "44": 1.0
  },
  "target_macro_f1_ge_0.85": true,
  "gain_over_baseline_pp": 0.0,
  "target_gain_ge_10pp": false
}
[robust] G1_customcnn-scratch_augon_s42 rotation    s1 F1 1.0000
[robust] G1_customcnn-scratch_augon_s42 rotation    s2 F1 1.0000
[robust] G1_customcnn-scratch_augon_s42 rotation    s3 F1 1.0000
[robust] G1_customcnn-scratch_augon_s42 perspective s1 F1 1.0000
[robust] G1_customcnn-scratch_augon_s42 perspective s2 F1 1.0000
[robust] G1_customcnn-scratch_augon_s42 perspective s3 F1 1.0000
[robust] G1_customcnn-scratch_augon_s42 elastic     s1 F1 1.0000
[robust] G1_customcnn-scratch_augon_s42 elastic     s2 F1 1.0000
[robust] G1_customcnn-scratch_augon_s42 elastic     s3 F1 1.0000
[robust] G1_customcnn-scratch_augon_s42 brightnes

In [20]:
# Error-analysis fallback (needs the latest code): pull, analyse the first model with test failures, copy outputs
!git -C /content/repo pull --ff-only
!python -m batik errors
!mkdir -p $DRIVE/outputs && cp -r artifacts/metrics artifacts/error_analysis artifacts/release figures $DRIVE/outputs/

Already up to date.
[errors] G3 final run G3_effb0-pt_augon_s42: 0 test errors
[errors] G3g final run G3g_effb0-pt_augon_s42: 0 test errors
[errors] G1 final run G1_customcnn-scratch_augon_s42: 0 test errors
[errors] G1g final run G1g_customcnn-scratch_augon_s42: 64 test errors
[errors] case 1: Asem Warak -> Kembang Sepatu (0.94) fixed_by=[]
[errors] case 2: Asem Warak -> Kembang Sepatu (0.92) fixed_by=[]
[errors] case 3: Asem Warak -> Kembang Sepatu (0.80) fixed_by=[]
[errors] case 4: Asem Warak -> Kembang Sepatu (0.77) fixed_by=[]
[errors] case 5: Asem Warak -> Kembang Sepatu (0.76) fixed_by=[]
[errors] case 6: Asem Warak -> Kembang Sepatu (0.74) fixed_by=[]
[errors] case 7: Asem Warak -> Kembang Sepatu (0.71) fixed_by=[]
[errors] case 8: Asem Warak -> Kembang Sepatu (0.71) fixed_by=[]
[errors] case 9: Asem Warak -> Kembang Sepatu (0.69) fixed_by=[]
[errors] case 10: Asem Warak -> Kembang Sepatu (0.68) fixed_by=[]
